# Agent 研究：指标、训练数据与评分合同
Python 3.10+，CPU，标准库，无网络或模型。选择对应章节运行；数值来自原创 fixture，不是 DSPy / GAIA / τ / RAGEN / Search-R1 的 benchmark 或训练成绩。先预测，修改一个失败样例，再记录自己的解释。

In [ ]:
checks={}

## 1. PPO 裁剪与 GRPO 组优势
注意负优势的 min、当前/旧策略比率和零方差。此处没有神经网络、KL 或梯度更新。

In [ ]:
from statistics import mean,pvariance
from math import sqrt
def clipped_objective(r,a,epsilon=.2):
    return min(r*a,max(1-epsilon,min(1+epsilon,r))*a)
assert abs(clipped_objective(1.4,1)-1.2)<1e-12
assert abs(clipped_objective(.6,-1)+.8)<1e-12
def relative_advantages(rewards):
    std=sqrt(pvariance(rewards))
    return [(r-mean(rewards))/(std+1e-8) for r in rewards]
assert relative_advantages([0,0,0])==[0,0,0]
assert relative_advantages([1,1,1])==[0,0,0]
assert relative_advantages([0,0,1])[-1]>0
checks['clipped_objective']=True
checks['zero_variance']=True
print('正/负优势:',clipped_objective(1.4,1),clipped_objective(.6,-1),'组优势:',relative_advantages([0,0,1]))

## 2. 动作 mask 与 rollout 身份
工具观察作为条件，但不是模型采样动作。按 rollout 身份归并，不能按并发到达次序拼接。

In [ ]:
segments=[('query',4),('observation',7),('answer',3),('padding',2)]
action=[int(kind in {'query','answer'}) for kind,n in segments for _ in range(n)]
attention=[int(kind!='padding') for kind,n in segments for _ in range(n)]
assert sum(action)==7 and sum(attention)==14
assert action[4:11]==[0]*7 and attention[4:11]==[1]*7
events=[{'rollout':'a','kind':'action','value':'query-a'},
        {'rollout':'b','kind':'action','value':'query-b'},
        {'rollout':'b','kind':'observation','value':'result-b'},
        {'rollout':'a','kind':'observation','value':'result-a'}]
grouped={}
for e in events:grouped.setdefault(e['rollout'],[]).append(e['value'])
assert grouped=={'a':['query-a','result-a'],'b':['query-b','result-b']}
checks['action_observation_mask']=True
checks['rollout_identity']=True
print('动作/上下文 token:',sum(action),sum(attention),'归属:',grouped)

## 3. τ 可靠性组合估计
n 次试验成功 c 次，不放回选择 k 次，计算至少一次与全部成功。每题计算，不能拿全局均值替代。

In [ ]:
from math import comb
def reliability(n,c,k):
    if not 0<=c<=n or not 1<=k<=n:raise ValueError('counts')
    all_ok=comb(c,k)/comb(n,k) if c>=k else 0.
    any_ok=1-(comb(n-c,k)/comb(n,k) if n-c>=k else 0.)
    return {'pass_at_k':any_ok,'pass_all_k':all_ok}
r=reliability(4,2,2)
assert abs(r['pass_at_k']-5/6)<1e-12 and abs(r['pass_all_k']-1/6)<1e-12
assert reliability(4,0,2)=={'pass_at_k':0.,'pass_all_k':0.}
assert reliability(4,4,2)=={'pass_at_k':1.,'pass_all_k':1.}
checks['reliability_estimators']=True
print(r)

## 4. 原创附件 join 与可信业务评分
商品 ID join，用 Decimal；缺失/重复键拒绝。评分器实际验结果，不把 exit 0、关键词或自述当正确。

In [ ]:
from decimal import Decimal
def total(prices,quantities):
    by_id={}
    for row in prices:
        if row['id'] in by_id:raise ValueError('duplicate price ID')
        price=Decimal(row['price'])
        if not price.is_finite() or price<0:raise ValueError('price')
        by_id[row['id']]=price
    result=Decimal(0)
    for row in quantities:
        if row['id'] not in by_id:raise ValueError('missing price')
        if type(row['quantity']) is not int or row['quantity']<0:raise ValueError('quantity')
        result+=by_id[row['id']]*row['quantity']
    return result
p=[{'id':'a','name':'同名商品','price':'0.10'},{'id':'b','name':'同名商品','price':'0.20'}]
q=[{'id':'a','quantity':3},{'id':'b','quantity':2}]
assert total(p,q)==Decimal('0.70')
assert total(p,[{'id':'a','quantity':0}])==0
for pp,qq in [(p,[{'id':'c','quantity':1}]),(p+p[:1],q),(p,[{'id':'a','quantity':True}])]:
    try: total(pp,qq)
    except ValueError:pass
    else:raise AssertionError('bad input accepted')
def grader(output):
    try:return Decimal(output)==Decimal('0.70')
    except ArithmeticError:return False
assert grader('0.7') and not grader('0.8') and not grader('fixed')
checks['attachment_contract']=True
print('真实 ID 连接总额:',total(p,q))

## 5. 评分器负例与优化数据划分
关键词得分可被无依据输出利用；严格标签/引用 ID 检查仍不能替代真实证据支持。本文没有运行 DSPy 优化器。

In [ ]:
def metric(prediction,gold,valid_sources):
    return prediction['answer']==gold and bool(prediction['citations']) and set(prediction['citations'])<=valid_sources
assert metric({'answer':'checkpoint','citations':['doc-a']},'checkpoint',{'doc-a'})
for bad in [{'answer':'checkpoint','citations':[]},
            {'answer':'checkpoint','citations':['fake']},
            {'answer':'wrong checkpoint','citations':['doc-a']}]:
    assert not metric(bad,'checkpoint',{'doc-a'})
splits={'train':{'source-a'},'val':{'source-b'},'test':{'source-c'}}
assert not (splits['train']&splits['val'] or splits['train']&splits['test'] or splits['val']&splits['test'])
checks['metric_negative_cases']=True
print('评分器负例拒绝；来源组无跨集合')

## 6. ReAct 停止与无信息观察
固定只读循环演示预算/证据不足的终止，不是模型推理复现。重复观察不会自动产生新知识。

In [ ]:
def research(observations,budget):
    seen=set()
    for i,observation in enumerate(observations[:budget],1):
        if observation in seen:return {'reason':'no_new_evidence','calls':i}
        seen.add(observation)
        if observation=='verified-answer':return {'reason':'completed','calls':i}
    return {'reason':'budget_exceeded','calls':min(budget,len(observations))}
assert research(['irrelevant','irrelevant','verified-answer'],3)['reason']=='no_new_evidence'
assert research(['new-a','new-b','verified-answer'],2)['reason']=='budget_exceeded'
assert research(['new-a','verified-answer'],3)['reason']=='completed'
checks['loop_termination']=True
print(research(['irrelevant','irrelevant'],4))

## 7. Jev 回退代价
已校准正确概率是前提，Choice confidence 不能直接当它。数字只说明期望损失合同。

In [ ]:
from decimal import Decimal
def decide(q,error_cost=20,fallback_cost=1):
    if not 0<=q<=1:raise ValueError('probability')
    return 'auto' if (Decimal('1')-Decimal(str(q)))*Decimal(str(error_cost))<Decimal(str(fallback_cost)) else 'fallback'
assert decide(.94)=='fallback' and decide(.96)=='auto' and decide(.95)=='fallback'
checks['risk_cost']=True
print(decide(.94),decide(.96))

## 8. 导回自己的学习记录
my_explanation 写预测、修改和观察。所有 checks 都来自上述 fixture 断言，无模型、训练或 benchmark 运行。导入不认证来源；失败结果不得改写成成功。

In [ ]:
from pathlib import Path
import json
my_explanation=''
report={'experiment':'agent-research-checks','scope':'stdlib fixtures; no model/service','checks':checks,'my_explanation':my_explanation}
assert len(checks)==9 and all(checks.values())
Path('mechanism_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
print(json.dumps(report,ensure_ascii=False,indent=2))